# Large-Scale Monte Carlo for Structure Formation from Lattice Defects

10 million defects, 256³ grid — production run for P(k) power spectrum.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import time

# ====================== PARAMETERS ======================
N_defects = 10_000_000      # 10 million defects
box_size = 1000.0           # Mpc simulation volume
grid_size = 256             # 256³ grid (good balance)
defect_size = 5.0           # Mpc correlation length
seed = 42

print(f"Starting simulation: {N_defects:,} defects on {grid_size}³ grid")
start_time = time.time()

# ====================== GENERATE DEFECTS ======================
np.random.seed(seed)
positions = np.random.uniform(0, box_size, (N_defects, 3))

# Add clustering (filamentary structure)
for i in tqdm(range(5000), desc="Adding clustering"):
    idx = np.random.randint(0, N_defects)
    positions[idx] += np.random.normal(0, defect_size, 3)
    positions[idx] = np.clip(positions[idx], 0, box_size)

# ====================== DENSITY FIELD ======================
print("Building density field...")
density = np.zeros((grid_size, grid_size, grid_size), dtype=np.float32)
for pos in tqdm(positions, desc="Binning defects"):
    idx = (pos / box_size * grid_size).astype(int)
    density[tuple(idx)] += 1.0

# ====================== POWER SPECTRUM ======================
print("Computing power spectrum...")
density_ft = np.fft.fftn(density)
power = np.abs(density_ft)**2
k = np.fft.fftfreq(grid_size, d=box_size/grid_size) * 2 * np.pi

# Radial average for 1D P(k)
k_mag = np.sqrt(k[:,None,None]**2 + k[None,:,None]**2 + k[None,None,:]**2)
k_bins = np.linspace(0, k.max(), 100)
p_k_1d = np.zeros(len(k_bins)-1)

for i in range(len(k_bins)-1):
    mask = (k_mag >= k_bins[i]) & (k_mag < k_bins[i+1])
    if mask.sum() > 0:
        p_k_1d[i] = power[mask].mean()

k_centers = (k_bins[:-1] + k_bins[1:]) / 2

print(f"Simulation completed in {time.time() - start_time:.1f} seconds")
print(f"Mean density: {density.mean():.4f}")

# ====================== PLOT ======================
plt.figure(figsize=(10,6))
plt.loglog(k_centers, p_k_1d, 'b-', label='P(k) from lattice defects')
plt.xlabel('k (Mpc^{-1})')
plt.ylabel('P(k)')
plt.title('Power Spectrum from 10M Lattice Defects')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.savefig('../figures/power-spectrum-10M.pdf')
plt.show()